# STEP — Sentinel-2 Collection-1 → GCS → Earth Engine ImageCollection

This notebook stages the prepared 2016–2017 `*_GEE.tif` files in Google Cloud Storage, creates the Earth Engine collection, builds ingestion manifests from the JSON sidecars, tests one image, then optionally bulk-ingests the full collection.

**Target EE collection**

`projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017`

Run through the single-image QA before enabling bulk ingestion.


In [1]:
%pip install -q --upgrade earthengine-api google-cloud-storage google-auth tqdm pandas rasterio

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [1]:
from pathlib import Path
import json, re, subprocess, time
from datetime import datetime, timezone

import pandas as pd
import rasterio
from tqdm.auto import tqdm
import ee
from google.cloud import storage
import google.auth
from google.auth.exceptions import DefaultCredentialsError

PROJECT_ID = "bop-nca-data-space"
EE_COLLECTION_ID = "projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017"

# Bucket names are globally unique. Change this only if Google says it is unavailable.
GCS_BUCKET = "bop-nca-data-space-s2-c1-staging"
GCS_LOCATION = "us-central1"
GCS_PREFIX = "s2-c1-2016-2017/geotiffs"

GEE_READY_DIR = Path(r"A:\NCA_DATA\S2_2016-2017\gee_ready")
MANIFEST_DIR = Path(r"A:\NCA_DATA\S2_2016-2017\ee_manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12","SCL"]
EXPECTED_COUNT = 562
UPLOAD_CHUNK_SIZE = 64 * 1024 * 1024

print(PROJECT_ID)
print(EE_COLLECTION_ID)
print(GEE_READY_DIR)


bop-nca-data-space
projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017
A:\NCA_DATA\S2_2016-2017\gee_ready


## 1. Authenticate Earth Engine and Google Cloud

Earth Engine uses `ee.Authenticate()`. Cloud Storage uses Application Default Credentials (ADC). If ADC are absent, the notebook opens the Google login through `gcloud` from inside Jupyter.


In [2]:
import ee

PROJECT_ID = "bop-nca-data-space"

# Force Earth Engine to discard/rewrite its stored user authorization.
ee.Authenticate(
    auth_mode="localhost",
    force=True,
)

ee.Initialize(
    project=PROJECT_ID
)

print(
    ee.String(
        "Earth Engine connected"
    ).getInfo()
)


Successfully saved authorization token.
Earth Engine connected


In [4]:
from pathlib import Path
import subprocess
import google.auth
from google.auth.exceptions import DefaultCredentialsError


GCLOUD_CMD = Path(
    r"C:\Users\scottfordham\AppData\Local\Google\Cloud SDK"
    r"\google-cloud-sdk\bin\gcloud.cmd"
)

print("gcloud executable:", GCLOUD_CMD)
print("Exists:", GCLOUD_CMD.exists())


def ensure_adc(project_id):
    try:
        creds, quota_project = google.auth.default()

        print(
            "ADC found. Quota project:",
            quota_project,
        )

        return creds

    except DefaultCredentialsError:
        print(
            "No Application Default Credentials found."
        )

        if not GCLOUD_CMD.exists():
            raise FileNotFoundError(
                f"Could not find gcloud at:\n{GCLOUD_CMD}"
            )

        subprocess.run(
            [
                str(GCLOUD_CMD),
                "auth",
                "application-default",
                "login",
            ],
            check=True,
        )

        subprocess.run(
            [
                str(GCLOUD_CMD),
                "auth",
                "application-default",
                "set-quota-project",
                project_id,
            ],
            check=True,
        )

        creds, quota_project = google.auth.default()

        print(
            "ADC created. Quota project:",
            quota_project,
        )

        return creds

gcloud executable: C:\Users\scottfordham\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd
Exists: True


In [5]:
gcp_credentials = ensure_adc(
    PROJECT_ID
)

storage_client = storage.Client(
    project=PROJECT_ID,
    credentials=gcp_credentials,
)

print(
    "Cloud Storage client ready."
)

No Application Default Credentials found.
ADC created. Quota project: None
Cloud Storage client ready.


## 2. Create or connect to the staging bucket

In [7]:
bucket = storage_client.lookup_bucket(GCS_BUCKET)

if bucket is None:
    bucket = storage_client.bucket(GCS_BUCKET)
    bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    bucket = storage_client.create_bucket(bucket, location=GCS_LOCATION)
    print("Created:", f"gs://{GCS_BUCKET}")
else:
    print("Using existing:", f"gs://{GCS_BUCKET}", "| location:", bucket.location)


Created: gs://bop-nca-data-space-s2-c1-staging


## 3. Inventory and validate the prepared TIFF + JSON pairs

In [8]:
def mget(meta, *keys, required=True, default=None):
    for key in keys:
        if key in meta and meta[key] not in (None, ""):
            return meta[key]
    if required:
        raise KeyError(f"Missing required metadata field; tried {keys}")
    return default

records = []
errors = []

for tif_path in tqdm(sorted(GEE_READY_DIR.glob("*_GEE.tif")), desc="Validating"):
    json_path = tif_path.with_suffix(".json")

    if not json_path.exists():
        errors.append({"file": tif_path.name, "error": "Missing JSON sidecar"})
        continue

    try:
        meta = json.loads(json_path.read_text(encoding="utf-8"))

        with rasterio.open(tif_path) as src:
            if src.count != 11:
                raise ValueError(f"Expected 11 bands; found {src.count}")
            if list(src.descriptions) != EXPECTED_BANDS:
                raise ValueError(f"Unexpected band order: {src.descriptions}")
            if any(d != "uint16" for d in src.dtypes):
                raise ValueError(f"Unexpected dtypes: {src.dtypes}")

        records.append({
            "product_id": mget(meta, "product_id", "PRODUCT_ID"),
            "sensing_time": mget(meta, "sensing_time", "SENSING_TIME"),
            "mgrs_tile": mget(meta, "mgrs_tile", "MGRS_TILE"),
            "processing_baseline": mget(meta, "processing_baseline", "PROCESSING_BASELINE"),
            "satellite": mget(meta, "satellite", "SATELLITE"),
            "source": mget(meta, "source", "SOURCE", required=False, default="CDSE_COLLECTION1_L2A"),
            "tif_path": tif_path,
            "json_path": json_path,
            "file_name": tif_path.name,
            "file_size_bytes": tif_path.stat().st_size,
        })

    except Exception as exc:
        errors.append({"file": tif_path.name, "error": repr(exc)})

inventory = pd.DataFrame(records)

print("Valid products:", len(inventory))
print("Expected final:", EXPECTED_COUNT)
print("Validation errors:", len(errors))

if len(inventory):
    print("Prepared volume:", f"{inventory.file_size_bytes.sum()/1024**3:,.2f} GiB")
    display(inventory.head())

if errors:
    display(pd.DataFrame(errors).head(20))


Validating:   0%|          | 0/562 [00:00<?, ?it/s]

Valid products: 562
Expected final: 562
Validation errors: 0
Prepared volume: 146.82 GiB


,product_id,sensing_time,mgrs_tile,processing_baseline,satellite,source,tif_path,json_path,file_name,file_size_bytes
0,S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_2...,2016-01-03T18:51:22+00:00,11TNH,N0500,S2A,CDSE_COLLECTION1_L2A,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_2...,500419369
1,S2A_MSIL2A_20160103T185122_N0500_R070_T11TNJ_2...,2016-01-03T18:51:22+00:00,11TNJ,N0500,S2A,CDSE_COLLECTION1_L2A,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,S2A_MSIL2A_20160103T185122_N0500_R070_T11TNJ_2...,192111447
2,S2A_MSIL2A_20160103T185122_N0500_R070_T11TPH_2...,2016-01-03T18:51:22+00:00,11TPH,N0500,S2A,CDSE_COLLECTION1_L2A,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,S2A_MSIL2A_20160103T185122_N0500_R070_T11TPH_2...,43586823
3,S2A_MSIL2A_20160110T184122_N0500_R027_T11TNH_2...,2016-01-10T18:41:22+00:00,11TNH,N0500,S2A,CDSE_COLLECTION1_L2A,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,S2A_MSIL2A_20160110T184122_N0500_R027_T11TNH_2...,660185075
4,S2A_MSIL2A_20160110T184122_N0500_R027_T11TNJ_2...,2016-01-10T18:41:22+00:00,11TNJ,N0500,S2A,CDSE_COLLECTION1_L2A,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_...,S2A_MSIL2A_20160110T184122_N0500_R027_T11TNJ_2...,265670059


## 4. Upload all prepared TIFFs to GCS

This is restartable. Files already present with the same byte size are skipped.


In [9]:
def object_name(file_name):
    return f"{GCS_PREFIX}/{file_name}"

existing = {
    blob.name: int(blob.size or 0)
    for blob in storage_client.list_blobs(GCS_BUCKET, prefix=GCS_PREFIX + "/")
}

results = []

for _, row in tqdm(inventory.iterrows(), total=len(inventory), desc="Uploading"):
    local = Path(row.tif_path)
    remote_name = object_name(local.name)
    local_size = int(row.file_size_bytes)

    if existing.get(remote_name) == local_size:
        results.append({"file": local.name, "status": "skipped"})
        continue

    blob = bucket.blob(remote_name)
    blob.chunk_size = UPLOAD_CHUNK_SIZE

    try:
        blob.upload_from_filename(str(local), timeout=900)
        blob.reload()

        if int(blob.size or 0) != local_size:
            raise RuntimeError("Remote byte size does not match local file.")

        existing[remote_name] = local_size
        results.append({"file": local.name, "status": "uploaded"})

    except Exception as exc:
        results.append({"file": local.name, "status": "failed", "error": repr(exc)})

upload_df = pd.DataFrame(results)
display(upload_df.status.value_counts().rename_axis("status").reset_index(name="count"))

if (upload_df.status == "failed").any():
    display(upload_df.loc[upload_df.status == "failed"].head(20))


Uploading:   0%|          | 0/562 [00:00<?, ?it/s]

,status,count
0,uploaded,562


## 5. Create the empty Earth Engine ImageCollection

In [10]:
def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except Exception:
        return False

if asset_exists(EE_COLLECTION_ID):
    print("Collection already exists:", EE_COLLECTION_ID)
else:
    ee.data.createAsset({"type": "IMAGE_COLLECTION"}, EE_COLLECTION_ID)
    print("Created:", EE_COLLECTION_ID)


Created: projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017


## 6. Generate manifests with acquisition metadata

The sidecars provide everything needed for the cloud-probability join:

- `system:time_start` ← manifest `startTime`
- `MGRS_TILE`
- `PRODUCT_ID`
- `PROCESSING_BASELINE`
- `SATELLITE`
- source provenance

Reflectance bands use `MEAN` pyramiding; categorical `SCL` uses `SAMPLE`. Nodata remains `0`.


In [11]:
def rfc3339(value):
    dt = datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def safe_asset_name(value):
    return re.sub(r"[^A-Za-z0-9_-]", "_", str(value))

def make_manifest(row):
    pid = str(row.product_id)
    timestamp = rfc3339(row.sensing_time)
    asset_id = f"{EE_COLLECTION_ID}/{safe_asset_name(pid)}"
    uri = f"gs://{GCS_BUCKET}/{object_name(row.file_name)}"

    bands = []
    for i, band in enumerate(EXPECTED_BANDS):
        bands.append({
            "id": band,
            "tilesetId": "0",
            "tilesetBandIndex": i,
            "missingData": {"values": [0]},
            "pyramidingPolicy": "SAMPLE" if band == "SCL" else "MEAN",
        })

    return {
        "name": asset_id,
        "tilesets": [{
            "id": "0",
            "sources": [{"uris": [uri]}],
        }],
        "bands": bands,
        "startTime": timestamp,
        "endTime": timestamp,
        "properties": {
            "PRODUCT_ID": pid,
            "MGRS_TILE": str(row.mgrs_tile),
            "PROCESSING_BASELINE": str(row.processing_baseline),
            "SATELLITE": str(row.satellite),
            "source": str(row.source),
            "source_group": "historical_cdse",
        },
    }

manifest_rows = []

for _, row in inventory.iterrows():
    manifest = make_manifest(row)
    path = MANIFEST_DIR / (Path(row.file_name).stem + ".manifest.json")
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    manifest_rows.append({
        "product_id": row.product_id,
        "asset_id": manifest["name"],
        "manifest_path": path,
        "gcs_uri": manifest["tilesets"][0]["sources"][0]["uris"][0],
        "startTime": manifest["startTime"],
    })

manifest_df = pd.DataFrame(manifest_rows)
print("Manifests:", len(manifest_df))
display(manifest_df.head())


Manifests: 562


,product_id,asset_id,manifest_path,gcs_uri,startTime
0,S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_2...,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,A:\NCA_DATA\S2_2016-2017\ee_manifests\S2A_MSIL...,gs://bop-nca-data-space-s2-c1-staging/s2-c1-20...,2016-01-03T18:51:22Z
1,S2A_MSIL2A_20160103T185122_N0500_R070_T11TNJ_2...,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,A:\NCA_DATA\S2_2016-2017\ee_manifests\S2A_MSIL...,gs://bop-nca-data-space-s2-c1-staging/s2-c1-20...,2016-01-03T18:51:22Z
2,S2A_MSIL2A_20160103T185122_N0500_R070_T11TPH_2...,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,A:\NCA_DATA\S2_2016-2017\ee_manifests\S2A_MSIL...,gs://bop-nca-data-space-s2-c1-staging/s2-c1-20...,2016-01-03T18:51:22Z
3,S2A_MSIL2A_20160110T184122_N0500_R027_T11TNH_2...,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,A:\NCA_DATA\S2_2016-2017\ee_manifests\S2A_MSIL...,gs://bop-nca-data-space-s2-c1-staging/s2-c1-20...,2016-01-10T18:41:22Z
4,S2A_MSIL2A_20160110T184122_N0500_R027_T11TNJ_2...,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,A:\NCA_DATA\S2_2016-2017\ee_manifests\S2A_MSIL...,gs://bop-nca-data-space-s2-c1-staging/s2-c1-20...,2016-01-10T18:41:22Z


## 7. Confirm all manifest source files exist in GCS

In [12]:
staging_errors = []

for _, row in inventory.iterrows():
    name = object_name(row.file_name)
    remote_size = existing.get(name)

    if remote_size is None:
        staging_errors.append({"file": row.file_name, "error": "Missing from GCS"})
    elif int(remote_size) != int(row.file_size_bytes):
        staging_errors.append({
            "file": row.file_name,
            "error": f"Size mismatch: local={row.file_size_bytes}, remote={remote_size}",
        })

print("Staging errors:", len(staging_errors))
if staging_errors:
    display(pd.DataFrame(staging_errors).head(20))
else:
    print("All inventoried TIFFs are staged and byte-size verified.")


Staging errors: 0
All inventoried TIFFs are staged and byte-size verified.


## 8. Ingest one test image

Run this before the bulk submission.


In [13]:
TEST_INDEX = 0

test_manifest_path = Path(manifest_df.iloc[TEST_INDEX].manifest_path)
test_manifest = json.loads(test_manifest_path.read_text(encoding="utf-8"))
test_asset_id = test_manifest["name"]

print("Test asset:", test_asset_id)

if asset_exists(test_asset_id):
    print("Test asset already exists; no new task submitted.")
    test_operation_name = None
else:
    response = ee.data.startIngestion(None, test_manifest)
    print("Response:", response)
    test_operation_name = response.get("name")
    print("Operation:", test_operation_name)


Test asset: projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017/S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631
Response: {'id': 'ZW3BW6FHERR6UWUBWCSCDKU6', 'name': 'projects/bop-nca-data-space/operations/ZW3BW6FHERR6UWUBWCSCDKU6', 'started': 'OK'}
Operation: projects/bop-nca-data-space/operations/ZW3BW6FHERR6UWUBWCSCDKU6


In [14]:
def wait_for_operation(operation_name, poll_seconds=15):
    if not operation_name:
        return None

    while True:
        op = ee.data.getOperation(operation_name)
        meta = op.get("metadata", {})
        state = meta.get("state", "UNKNOWN")
        progress = meta.get("progress")

        print(
            f"{state}"
            + (f" | {progress:.1%}" if isinstance(progress, (int, float)) else "")
        )

        if op.get("done"):
            if op.get("error"):
                print("FAILED:", op["error"])
            else:
                print("COMPLETED")
            return op

        time.sleep(poll_seconds)

if test_operation_name:
    test_operation = wait_for_operation(test_operation_name)


PENDING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
SUCCEEDED
COMPLETED


## 9. QA the ingested test asset

In [15]:
if not asset_exists(test_asset_id):
    raise RuntimeError("Test asset is not available yet.")

img = ee.Image(test_asset_id)
info = img.getInfo()

print("Bands:", [b["id"] for b in info["bands"]])
print("Date:", ee.Date(img.get("system:time_start")).format("YYYY-MM-dd HH:mm:ss").getInfo())
print("Properties:")
display(pd.Series(info.get("properties", {})))


Bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL']
Date: 2016-01-03 18:51:22
Properties:


SATELLITE                                                            S2A
system:time_end                                            1451847082000
source                                              CDSE_COLLECTION1_L2A
PRODUCT_ID             S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_2...
system:time_start                                          1451847082000
source_group                                             historical_cdse
PROCESSING_BASELINE                                                N0500
system:footprint       {'type': 'LinearRing', 'coordinates': [[-115.6...
system:asset_size                                              734381288
MGRS_TILE                                                          11TNH
system:index           S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_2...
dtype: object

## 10. QA the cloud-probability join for the test image

The custom image joins on **MGRS tile + acquisition time**. The cloud tile code is parsed from the cloud image `system:index`, so the workflow does not depend on a native `MGRS_TILE` property being present there.


In [16]:
def add_parsed_cloud_tile(image):
    image = ee.Image(image)
    parts = ee.String(image.get("system:index")).split("_")
    last = ee.String(parts.get(parts.length().subtract(1)))
    return image.set("MGRS_TILE_PARSED", last.slice(1))

test_date = ee.Date(img.get("system:time_start"))

cloud_candidates = (
    ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
    .filterDate(test_date.advance(-10, "minute"), test_date.advance(10, "minute"))
    .filterBounds(img.geometry())
    .map(add_parsed_cloud_tile)
)

print("Cloud candidates:", cloud_candidates.size().getInfo())
print("Candidate IDs:", cloud_candidates.aggregate_array("system:index").getInfo())

join_filter = ee.Filter.And(
    ee.Filter.equals(leftField="MGRS_TILE", rightField="MGRS_TILE_PARSED"),
    ee.Filter.maxDifference(
        difference=5 * 60 * 1000,
        leftField="system:time_start",
        rightField="system:time_start",
    ),
)

joined = ee.ImageCollection(
    ee.Join.saveFirst("cloudprob").apply(
        primary=ee.ImageCollection([img]),
        secondary=cloud_candidates,
        condition=join_filter,
    )
)

joined_img = ee.Image(joined.first())
cloud_match = joined_img.get("cloudprob").getInfo()

print("Historical tile:", img.get("MGRS_TILE").getInfo())
print("Cloud match found:", cloud_match is not None)


Cloud candidates: 8
Candidate IDs: ['20160103T185122_20160103T185116_T11TNH', '20160103T185122_20160103T185116_T11TNJ', '20160103T185122_20160103T185116_T11TPH', '20160103T185122_20160103T185116_T11TPJ', '20160103T185246_20160119T162332_T11TNH', '20160103T185246_20160119T162332_T11TNJ', '20160103T185246_20160119T162332_T11TPH', '20160103T185246_20160119T162332_T11TPJ']
Historical tile: 11TNH
Cloud match found: True


## 11. Bulk ingestion

Only change `RUN_BULK_INGEST` to `True` after the test image and cloud join both pass.

The loop is restartable: existing Earth Engine assets are skipped.


In [17]:
RUN_BULK_INGEST = True
SUBMIT_PAUSE_SECONDS = 2.0

bulk_results = []

if not RUN_BULK_INGEST:
    print("Bulk ingestion disabled.")
else:
    for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Submitting"):
        manifest = json.loads(Path(row.manifest_path).read_text(encoding="utf-8"))
        asset_id = manifest["name"]

        if asset_exists(asset_id):
            bulk_results.append({
                "product_id": row.product_id,
                "status": "skipped_existing",
                "operation": None,
                "error": None,
            })
            continue

        try:
            response = ee.data.startIngestion(None, manifest)
            bulk_results.append({
                "product_id": row.product_id,
                "status": "submitted",
                "operation": response.get("name"),
                "error": None,
            })
        except Exception as exc:
            bulk_results.append({
                "product_id": row.product_id,
                "status": "submission_failed",
                "operation": None,
                "error": repr(exc),
            })

        time.sleep(SUBMIT_PAUSE_SECONDS)

    bulk_results_df = pd.DataFrame(bulk_results)
    display(bulk_results_df.status.value_counts().rename_axis("status").reset_index(name="count"))

    log_path = MANIFEST_DIR / "bulk_ingestion_submissions.csv"
    bulk_results_df.to_csv(log_path, index=False)
    print("Log:", log_path)


Submitting:   0%|          | 0/562 [00:00<?, ?it/s]

,status,count
0,submitted,561
1,skipped_existing,1


Log: A:\NCA_DATA\S2_2016-2017\ee_manifests\bulk_ingestion_submissions.csv


## 12. Monitor recent Earth Engine operations

In [ ]:
ops = ee.data.listOperations(1000)

rows = []
for op in ops:
    meta = op.get("metadata", {})
    rows.append({
        "name": op.get("name"),
        "state": meta.get("state"),
        "type": meta.get("type"),
        "progress": meta.get("progress"),
        "done": op.get("done", False),
        "error": op.get("error", {}).get("message") if op.get("error") else None,
    })

operations_df = pd.DataFrame(rows)

if len(operations_df):
    display(operations_df.state.value_counts(dropna=False).rename_axis("state").reset_index(name="count"))
    display(operations_df.head(30))
else:
    print("No recent operations returned.")


## 13. Final collection QA

Run after all ingestion jobs finish. Expected final collection size: **562**.


In [20]:
historical = ee.ImageCollection(EE_COLLECTION_ID)

print("Total:", historical.size().getInfo())
print("2016:", historical.filterDate("2016-01-01", "2017-01-01").size().getInfo())
print("2017:", historical.filterDate("2017-01-01", "2018-01-01").size().getInfo())
print("By tile:", historical.aggregate_histogram("MGRS_TILE").getInfo())

first = ee.Image(historical.sort("system:time_start").first())
print("First date:", ee.Date(first.get("system:time_start")).format("YYYY-MM-dd HH:mm:ss").getInfo())
print("First bands:", first.bandNames().getInfo())


Total: 107
2016: 107
2017: 0
By tile: {'11TNH': 29, '11TNJ': 36, '11TPH': 42}
First date: 2016-01-03 18:51:22
First bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL']


## 14. Full 2016–2017 cloud-probability join QA

In [ ]:
historical = ee.ImageCollection(EE_COLLECTION_ID)

clouds = (
    ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
    .filterDate("2016-01-01", "2018-01-01")
    .map(add_parsed_cloud_tile)
)

historical_join_filter = ee.Filter.And(
    ee.Filter.equals(leftField="MGRS_TILE", rightField="MGRS_TILE_PARSED"),
    ee.Filter.maxDifference(
        difference=5 * 60 * 1000,
        leftField="system:time_start",
        rightField="system:time_start",
    ),
)

joined_raw = ee.ImageCollection(
    ee.Join.saveFirst("cloudprob").apply(
        primary=historical,
        secondary=clouds,
        condition=historical_join_filter,
    )
)

matched = joined_raw.filter(ee.Filter.notNull(["cloudprob"]))

total = historical.size()
matched_n = matched.size()

print("Historical:", total.getInfo())
print("Matched:", matched_n.getInfo())
print("Missing:", total.subtract(matched_n).getInfo())
print("Matched by tile:", matched.aggregate_histogram("MGRS_TILE").getInfo())


## 15. Optional staging cleanup

Do not enable this until the Earth Engine collection and cloud join have both been fully verified.


In [ ]:
DELETE_STAGED_GEOTIFFS = False

if not DELETE_STAGED_GEOTIFFS:
    print("Cleanup disabled.")
else:
    blobs = list(storage_client.list_blobs(GCS_BUCKET, prefix=GCS_PREFIX + "/"))
    print("Deleting:", len(blobs))
    for blob in tqdm(blobs, desc="Deleting staged TIFFs"):
        blob.delete()
    print("Cleanup complete.")
